# Baseline — IOAI 2026 Home Task 2: Gridworld Delivery Agent

**Competition:** an agent lives on an 8×8 grid with walls and 6 depots. Each episode:
go to the depot holding the **package**, `pickup`, carry it to the **destination**
depot, `dropoff`. `train_demos.pkl` holds 400 expert trajectories
(observations + actions); `valid_scenarios.pkl` / `test_scenarios.pkl` hold scenario
specs. Actions: 0=south, 1=north, 2=east, 3=west, 4=pickup, 5=dropoff.

- **Kaggle link:** _TODO: add link_

Two baselines here:
1. **Behavior cloning (ML)** — supervised action prediction from observations;
2. **BFS planner (search)** — scenarios expose the full state (walls, depots,
   positions), so shortest-path planning solves an episode outright and makes a
   strong reference solution.

In [1]:
import pickle
import numpy as np

DATA_DIR = "."
demos = pickle.load(open(f"{DATA_DIR}/train_demos.pkl", "rb"))
valid = pickle.load(open(f"{DATA_DIR}/valid_scenarios.pkl", "rb"))
test  = pickle.load(open(f"{DATA_DIR}/test_scenarios.pkl", "rb"))
cfg = demos["environment_config"]
print(cfg)
print(len(demos["trajectories"]), "demo trajectories;",
      len(valid), "valid /", len(test), "test scenarios")

{'grid_size': 8, 'n_depots': 6, 'n_walls': 8, 'max_steps': 120, 'action_names': {0: 'south', 1: 'north', 2: 'east', 3: 'west', 4: 'pickup', 5: 'dropoff'}}
400 demo trajectories; 200 valid / 1600 test scenarios


In [2]:
# ---------- Baseline 1: behavior cloning ----------
# Flatten each observation (grid planes + vector) and train a classifier to imitate
# the expert's action.
X, y = [], []
for t in demos["trajectories"]:
    for obs, a in zip(t["observations"], t["actions"]):
        X.append(np.concatenate([np.asarray(obs["grid"]).ravel(),
                                 np.asarray(obs["vector"]).ravel()]))
        y.append(a)
X, y = np.array(X, dtype=np.float32), np.array(y)
print(X.shape, np.bincount(y))

(5327, 397) [1066 1148 1120 1193  400  400]


In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

Xtr, Xva, ytr, yva = train_test_split(X, y, test_size=0.2, random_state=0)
clf = LogisticRegression(max_iter=1000)
clf.fit(Xtr, ytr)
print(f"held-out action accuracy: {clf.score(Xva, yva):.4f}")
# NB: high action accuracy != task success; compounding errors matter (see ideas).

held-out action accuracy: 0.7645


In [4]:
# ---------- Baseline 2: BFS shortest-path planner ----------
from collections import deque

MOVES = {0: (1, 0), 1: (-1, 0), 2: (0, 1), 3: (0, -1)}   # south,north,east,west (row,col)

def bfs_path(start, goal, walls, n):
    if start == goal: return []
    prev = {start: None}
    dq = deque([start])
    while dq:
        cur = dq.popleft()
        for a, (dr, dc) in MOVES.items():
            nxt = (cur[0] + dr, cur[1] + dc)
            if (0 <= nxt[0] < n and 0 <= nxt[1] < n
                    and nxt not in walls and nxt not in prev):
                prev[nxt] = (cur, a)
                if nxt == goal:
                    acts = []
                    node = nxt
                    while prev[node] is not None:
                        node, a = prev[node]
                        acts.append(a)
                    return acts[::-1]
                dq.append(nxt)
    return None

def solve(scn, n=8):
    walls = set(map(tuple, scn["walls"]))
    depots = list(map(tuple, scn["depots"]))
    agent = tuple(scn["agent_pos"])
    pkg, dst = depots[scn["package_location"]], depots[scn["destination"]]
    p1 = bfs_path(agent, pkg, walls, n)
    p2 = bfs_path(pkg, dst, walls, n)
    if p1 is None or p2 is None: return [0]           # unreachable: dummy
    return p1 + [4] + p2 + [5]                        # ...pickup...dropoff

plans = [solve(s) for s in valid]
lens = [len(p) for p in plans]
print(f"valid scenarios planned: {len(plans)}, mean plan length {np.mean(lens):.1f}, "
      f"max {max(lens)} (limit {cfg['max_steps']})")

valid scenarios planned: 200, mean plan length 13.3, max 26 (limit 120)


In [5]:
# Plan the test scenarios and save action sequences
import pandas as pd
test_plans = [solve(s) for s in test]
sub = pd.DataFrame({
    "layout_id": [s["layout_id"] for s in test],
    "actions": [" ".join(map(str, p)) for p in test_plans],
})
sub.to_csv("submission.csv", index=False)   # adapt columns to the required format
sub.head()

,layout_id,actions
0,test_0000,1 2 2 0 4 1 1 3 3 5
1,test_0000,1 2 2 2 4 1 3 3 3 3 3 3 3 0 5
2,test_0000,0 0 4 0 0 0 2 2 2 0 0 2 2 5
3,test_0000,1 1 3 3 3 3 1 3 3 3 1 1 4 0 0 0 0 0 5
4,test_0001,0 3 4 1 5


## Ideas to improve

- Roll out the behavior-cloning policy in the environment and measure **episode
  success**, not just action accuracy — then fix its compounding errors with DAgger
  (query the BFS expert on the states the policy visits).
- Replace the linear model with a small CNN over the grid planes.
- The planner is optimal for this deterministic task — the ML exercise is making a
  *learned* policy match it. Compare both on the validation scenarios.
